# ABC2Vec — Notebook 11: Probing Experiments

**What do embedding dimensions encode?**

Probing classifiers test whether specific musical properties are linearly decodable
from the embedding space.

**Probes:**
1. **Mode** — Dorian vs. Mixolydian vs. Major vs. Minor
2. **Tune type / Rhythm** — Jig (6/8) vs. Reel (4/4) vs. Hornpipe vs. Polka
3. **Key root** — D, G, A, C, etc.
4. **Time signature** — 4/4, 6/8, 3/4, etc.
5. **Regional origin** — if labels available
6. **Tune length** — short vs. medium vs. long
7. **Dimensionality analysis** — which embedding dimensions are most informative?

In [ ]:
import os, sys, json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.decomposition import PCA

PROJECT_DIR = Path('/Volumes/LLModels/ABC2VEC')
PROCESSED_DIR = PROJECT_DIR / 'data' / 'processed'
RESULTS_DIR = PROJECT_DIR / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(PROJECT_DIR))

device = torch.device('cuda' if torch.cuda.is_available() else 
                       'mps' if torch.backends.mps.is_available() else 'cpu')

## 11.1 Load Model and Encode Full Test Set

In [ ]:
from abc2vec.model import ABC2VecModel, ABC2VecConfig
from abc2vec.tokenizer import ABCVocabulary, BarPatchifier

vocab = ABCVocabulary()
vocab_path = PROJECT_DIR / 'data' / 'processed' / 'vocabulary.json'
if vocab_path.exists():
    vocab.load(vocab_path)

patchifier = BarPatchifier(vocab, max_bar_length=64, max_bars=64)

config = ABC2VecConfig(
    vocab_size=len(vocab),
    max_bar_len=64, max_bars=64,
    d_model=256, n_heads=8, n_layers=6,
    d_ff=1024, d_embed=128, dropout=0.1,
)
model = ABC2VecModel(config).to(device)

ckpt_path = PROJECT_DIR / 'checkpoints' / 'best_model.pt'
if ckpt_path.exists():
    checkpoint = torch.load(ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

# Load test set
test_df = pd.read_parquet(PROCESSED_DIR / 'test.parquet')
print(f"Test set: {len(test_df)} tunes")
print(f"Columns: {list(test_df.columns)}")

In [ ]:
@torch.no_grad()
def encode_tunes(abc_texts, model, patchifier, device, batch_size=64):
    model.eval()
    all_emb = []
    for i in tqdm(range(0, len(abc_texts), batch_size), desc="Encoding"):
        batch = abc_texts[i:i+batch_size]
        bar_ids_list, bar_mask_list = [], []
        for text in batch:
            bars = patchifier.patchify(text)
            ids, mask = patchifier.encode(bars)
            bar_ids_list.append(ids)
            bar_mask_list.append(mask)
        bar_ids = torch.stack(bar_ids_list).to(device)
        bar_mask = torch.stack(bar_mask_list).to(device)
        emb = model.get_embedding(bar_ids, bar_mask)
        all_emb.append(emb.cpu().numpy())
    return np.concatenate(all_emb, axis=0)

print("Encoding test set...")
test_emb = encode_tunes(test_df['abc_body'].tolist(), model, patchifier, device)
print(f"Embeddings: {test_emb.shape}")

## 11.2 Helper: Run Probing Classifier

In [ ]:
def run_probe(embeddings: np.ndarray, labels: np.ndarray, property_name: str,
              min_class_size: int = 30, n_folds: int = 5) -> dict:
    """
    Run a linear probing experiment.
    
    Returns dict with accuracy, per-class metrics, and feature importances.
    """
    le = LabelEncoder()
    y = le.fit_transform(labels)
    classes = le.classes_
    
    # Filter rare classes
    class_counts = np.bincount(y)
    valid_classes = np.where(class_counts >= min_class_size)[0]
    mask = np.isin(y, valid_classes)
    X = embeddings[mask]
    y = y[mask]
    
    # Re-encode after filtering
    le2 = LabelEncoder()
    y = le2.fit_transform(y)
    classes = [classes[c] for c in valid_classes]
    
    print(f"\n{'='*50}")
    print(f"Probe: {property_name}")
    print(f"{'='*50}")
    print(f"  Samples: {len(X)}, Classes: {len(classes)}")
    print(f"  Classes: {classes}")
    
    # Cross-validated logistic regression
    clf = LogisticRegression(
        max_iter=3000, C=1.0, multi_class='multinomial',
        solver='lbfgs', class_weight='balanced'
    )
    
    cv = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)
    scores = cross_val_score(clf, X, y, cv=cv, scoring='accuracy')
    f1_scores = cross_val_score(clf, X, y, cv=cv, scoring='f1_weighted')
    
    # Chance level
    chance = 1.0 / len(classes)
    
    print(f"  Accuracy: {scores.mean():.4f} ± {scores.std():.4f}")
    print(f"  F1 (weighted): {f1_scores.mean():.4f} ± {f1_scores.std():.4f}")
    print(f"  Chance level: {chance:.4f}")
    print(f"  Above chance: +{scores.mean() - chance:.4f}")
    
    # Fit full model for feature importance
    clf.fit(X, y)
    
    # Feature importance: weight magnitudes across classes
    # Shape: (n_classes, d_embed)
    weights = np.abs(clf.coef_)  # (n_classes, d_embed)
    dim_importance = weights.mean(axis=0)  # Average across classes
    
    return {
        'property': property_name,
        'n_samples': len(X),
        'n_classes': len(classes),
        'classes': classes,
        'accuracy': scores.mean(),
        'accuracy_std': scores.std(),
        'f1': f1_scores.mean(),
        'f1_std': f1_scores.std(),
        'chance': chance,
        'dim_importance': dim_importance,
        'clf': clf,
    }

## 11.3 Probe 1: Mode (Dorian vs. Mixolydian vs. Major vs. Minor)

In [ ]:
if 'mode' in test_df.columns:
    mode_mask = test_df['mode'].notna()
    probe_mode = run_probe(
        test_emb[mode_mask.values],
        test_df.loc[mode_mask, 'mode'].values,
        'Mode'
    )
else:
    print("No 'mode' column available.")
    probe_mode = None

## 11.4 Probe 2: Tune Type (Rhythm)

In [ ]:
if 'tune_type' in test_df.columns:
    type_mask = test_df['tune_type'].notna()
    probe_type = run_probe(
        test_emb[type_mask.values],
        test_df.loc[type_mask, 'tune_type'].values,
        'Tune Type'
    )
else:
    print("No 'tune_type' column.")
    probe_type = None

## 11.5 Probe 3: Key Root

In [ ]:
if 'key_root' in test_df.columns:
    key_mask = test_df['key_root'].notna()
    probe_key = run_probe(
        test_emb[key_mask.values],
        test_df.loc[key_mask, 'key_root'].values,
        'Key Root'
    )
else:
    print("No 'key_root' column.")
    probe_key = None

## 11.6 Probe 4: Time Signature

In [ ]:
# Extract time signature from ABC header if not already parsed
import re

def extract_time_sig(abc_text: str) -> str | None:
    """Extract time signature from ABC notation."""
    # Look for M: field
    m = re.search(r'M:\s*(\d+/\d+)', abc_text)
    if m:
        return m.group(1)
    return None

if 'time_sig' not in test_df.columns:
    test_df['time_sig'] = test_df['abc_body'].apply(extract_time_sig)

ts_mask = test_df['time_sig'].notna()
if ts_mask.sum() > 100:
    probe_ts = run_probe(
        test_emb[ts_mask.values],
        test_df.loc[ts_mask, 'time_sig'].values,
        'Time Signature'
    )
else:
    print(f"Only {ts_mask.sum()} tunes with time signature — skipping.")
    probe_ts = None

## 11.7 Probe 5: Tune Length

In [ ]:
# Bin tune length into categories
lengths = test_df['abc_body'].str.len()
q33, q66 = lengths.quantile(0.33), lengths.quantile(0.66)

def length_bin(l):
    if l < q33:
        return 'short'
    elif l < q66:
        return 'medium'
    else:
        return 'long'

test_df['length_bin'] = lengths.apply(length_bin)

probe_length = run_probe(
    test_emb,
    test_df['length_bin'].values,
    'Tune Length'
)

## 11.8 Summary of All Probes

In [ ]:
all_probes = [p for p in [probe_mode, probe_type, probe_key, probe_ts, probe_length] if p is not None]

summary_data = []
for p in all_probes:
    summary_data.append({
        'Property': p['property'],
        'N Classes': p['n_classes'],
        'Accuracy': f"{p['accuracy']:.4f} ± {p['accuracy_std']:.4f}",
        'F1': f"{p['f1']:.4f} ± {p['f1_std']:.4f}",
        'Chance': f"{p['chance']:.4f}",
        'Delta': f"+{p['accuracy'] - p['chance']:.4f}",
    })

summary_df = pd.DataFrame(summary_data)
print("\nProbing Summary:")
print(summary_df.to_string(index=False))

summary_df.to_csv(RESULTS_DIR / 'probing_summary.csv', index=False)

# Bar chart
fig, ax = plt.subplots(figsize=(10, 5))
props = [p['property'] for p in all_probes]
accs = [p['accuracy'] for p in all_probes]
chances = [p['chance'] for p in all_probes]

x = np.arange(len(props))
width = 0.35
ax.bar(x - width/2, accs, width, label='Probe Accuracy', color='steelblue')
ax.bar(x + width/2, chances, width, label='Chance Level', color='lightgray')
ax.set_xticks(x)
ax.set_xticklabels(props, rotation=15, ha='right')
ax.set_ylabel('Accuracy')
ax.set_title('Linear Probing Accuracy vs. Chance')
ax.legend()
ax.set_ylim(0, 1)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'probing_summary.png', dpi=150, bbox_inches='tight')
plt.show()

## 11.9 Dimension Analysis

In [ ]:
# Which embedding dimensions are most informative for each property?
fig, axes = plt.subplots(len(all_probes), 1, figsize=(16, 4*len(all_probes)))
if len(all_probes) == 1:
    axes = [axes]

for i, probe in enumerate(all_probes):
    importance = probe['dim_importance']
    top_dims = np.argsort(importance)[::-1][:20]
    
    axes[i].bar(range(len(importance)), importance, color='steelblue', alpha=0.7)
    axes[i].set_xlabel('Embedding Dimension')
    axes[i].set_ylabel('Importance')
    axes[i].set_title(f'{probe["property"]} — Dimension Importance')
    
    # Highlight top dimensions
    for d in top_dims[:5]:
        axes[i].bar(d, importance[d], color='red', alpha=0.8)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'probing_dimension_importance.png', dpi=150, bbox_inches='tight')
plt.show()

# Cross-property dimension overlap
if len(all_probes) >= 2:
    print("\nTop-10 Dimensions Per Property:")
    top_10s = {}
    for probe in all_probes:
        top_10 = set(np.argsort(probe['dim_importance'])[::-1][:10])
        top_10s[probe['property']] = top_10
        print(f"  {probe['property']}: {sorted(top_10)}")
    
    print("\nOverlap between properties:")
    props = list(top_10s.keys())
    for i in range(len(props)):
        for j in range(i+1, len(props)):
            overlap = top_10s[props[i]] & top_10s[props[j]]
            print(f"  {props[i]} ∩ {props[j]}: {len(overlap)} dims {sorted(overlap)}")

## 11.10 PCA of Embedding Space

In [ ]:
# PCA analysis of the embedding space
pca = PCA(n_components=min(50, test_emb.shape[1]))
pca.fit(test_emb)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Explained variance
axes[0].plot(np.cumsum(pca.explained_variance_ratio_), 'o-', color='steelblue')
axes[0].set_xlabel('Number of Components')
axes[0].set_ylabel('Cumulative Explained Variance')
axes[0].set_title('PCA — Explained Variance')
axes[0].axhline(y=0.9, color='red', linestyle='--', label='90%')
axes[0].axhline(y=0.95, color='orange', linestyle='--', label='95%')
axes[0].legend()

# Individual component variance
axes[1].bar(range(len(pca.explained_variance_ratio_)),
            pca.explained_variance_ratio_, color='steelblue')
axes[1].set_xlabel('Component')
axes[1].set_ylabel('Explained Variance Ratio')
axes[1].set_title('PCA — Per-Component Variance')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'probing_pca.png', dpi=150, bbox_inches='tight')
plt.show()

# How many components for 90% / 95%?
cumvar = np.cumsum(pca.explained_variance_ratio_)
n_90 = np.argmax(cumvar >= 0.9) + 1
n_95 = np.argmax(cumvar >= 0.95) + 1
print(f"Components for 90% variance: {n_90}")
print(f"Components for 95% variance: {n_95}")
print(f"Effective dimensionality is ~{n_90}-{n_95} out of {test_emb.shape[1]}")